# A10

## ML - Learning Kinect 2D from PoseNet 2D

### 1. Task and objective

The goal of Lab 10 was to learn the Kinect 2D joint coordinates from PoseNet 2D joint coordinates. In the 2D PoseNet case, the lecture defines the task as mapping 13 joints × 2 dimensions from PoseNet, `(xpn, ypn)`, to 13 joints × 2 dimensions in Kinect, `(xk, yk)`, which gives 26 input features and 26 target features. The lecture also requires going through the Deep Learning steps, varying hyper-parameters, suggesting an optimal setup, using k-fold cross validation, early stopping, and evaluating on a user-meaningful metric such as deviation in centimeters. It additionally suggests trying a one-step variant that maps `(xpn, ypn)` directly to `(xk, yk, zk)` with one network.

This notebook focuses on the week 16 task, that is, learning Kinect 2D from PoseNet 2D. The main objective was therefore to build and evaluate neural-network regressors that predict Kinect 2D skeleton coordinates from PoseNet 2D skeleton coordinates, and then identify the best-performing setup.

### 2. Deep Learning steps

The work followed the Deep Learning workflow described in the lecture: load and prepare the paired Kinect/PoseNet frame sequences, define a network, compile the model, split training and testing data using k-fold cross validation, train with early stopping, evaluate on the test data, and finally save and reuse the trained model.

#### 2.1 Load and prepare movement data

The starting point was a set of Kinect CSV files and generated PoseNet CSV files from the same Kinect videos. The lecture states that the Kinect videos should be fed through PoseNet to obtain corresponding `(xpn, ypn)` joint positions per joint and frame, and then paired with the Kinect `(xk, yk, zk)` positions.

In the implemented pipeline:
- PoseNet CSV files were generated from the Kinect videos.
- Matching PoseNet and Kinect files were joined by sequence ID and frame number.
- The final 2D learning task used 26 PoseNet input features and 26 Kinect target features.
- After fixing the PoseNet generation and reducing dropped data, 179 matched sequences were available for model training and evaluation.

Additional preprocessing was used to make learning more stable:
- **Hip-centered normalization** was applied by subtracting the hip-center from each joint.
- **Torso scaling** was optionally applied by dividing coordinates by torso length.
- **Interpolation** was used to reduce the effect of missing values instead of discarding large parts of the data.
- **Mirroring augmentation** was used only on the training folds to increase effective training diversity while avoiding test leakage.

#### 2.2 Define Deep Learning models

The lecture presents the week 16 problem as a 26-to-26 regression problem and explicitly mentions testing Dense variants with one hidden layer of 32, 64, and 128 nodes.

To cover both the lecture baseline and more advanced temporal alternatives, the following model families were evaluated:
- **Dense**
- **Conv1D**
- **GRU**
- **LSTM**
- **TCN**

Dense served as the lecture-aligned baseline, while the sequence-aware models were added to test whether temporal context across frames improves prediction accuracy.

#### 2.3 Compile the models

The lecture requires a regression loss and recommends including MAE as a metric for explainability.

In this notebook:
- **Optimizer:** Adam
- **Loss:** MSE
- **Evaluation metrics:** mean test MAE and average 2D distance in cm

The raw training objective was therefore MSE, while the main human-interpretable evaluation metric was the average absolute error converted to centimeters.

#### 2.4 Split training and testing sets

The lecture requires k-fold cross validation and states that at least 10 random sequences should be used for testing.

This notebook used:
- **10-fold GroupKFold cross validation**
- grouping by sequence ID, so that sequences were kept separate across folds
- a validation split inside each training fold for early stopping

With 179 matched sequences and 10 folds, each test fold contained well above the lecture minimum of 10 random sequences.

#### 2.5 Train the models

The lecture explicitly requires early stopping to reach an appropriate number of epochs.

Training was therefore done with:
- mini-batch training in PyTorch
- early stopping based on validation loss
- GPU execution when CUDA was available
- fold-wise scaling fitted only on the training split

This avoided leakage from the validation and test sets into the scalers.

#### 2.6 Evaluate model predictions

The lecture states that evaluation should be done on the test data using a metric that makes sense for the user, and gives average deviation in cm as an example.

For that reason, the main reported results are:
- **cross-validated mean test MAE**
- **cross-validated mean test MAE in cm**
- **cross-validated average 2D joint distance in cm**
- **per-joint average distance in cm**

These values were computed on the held-out test fold in each cross-validation iteration and then averaged across folds.

#### 2.7 Use the model

The final champion model was retrained, saved, and reloaded together with the input and target scalers and a metadata file. This supports the lecture’s software-development requirement that the model should be easy to retrain and use.

---

### 3. Hyper-parameter exploration

A broad model-family screening was first carried out using Dense, Conv1D, GRU, LSTM, TCN, and Transformer variants. After that, a reduced screening round was run to identify the strongest model family more efficiently. GRU consistently emerged as the best family.

A focused tuning round was then performed around the GRU champion by varying:
- hidden size
- number of layers
- dropout
- bidirectionality
- learning rate
- training budget through higher epoch and patience limits

The best model from the GRU tuning round was:

**BiGRU_256_2layers_drop001_lr1e3**

with:
- **mean test MAE:** 0.038096
- **mean test MAE in cm:** 3.809565 cm
- **mean average 2D distance:** 6.160603 cm
- **mean best validation loss:** 0.287684
- **mean best epoch:** 16.3

This became the recommended setup.

---

### 4. Optimum settings: what worked best

The best-performing setup in the final tuning round was:

- **Model family:** GRU
- **Variant:** BiGRU_256_2layers_drop001_lr1e3
- **Hidden size:** 256
- **Layers:** 2
- **Bidirectional:** yes
- **Dropout:** 0.01
- **Learning rate:** 1e-3
- **Epoch budget:** increased relative to the first screening round
- **Early stopping:** enabled
- **Preprocessing:** hip-centered normalization, torso scaling, interpolation, fold-wise standardization, and training-only mirroring

The strongest overall pattern in the experiments was that sequence-aware recurrent models performed better than Dense and Conv1D models. GRU and LSTM were both clearly stronger than the lecture-style Dense baselines, and GRU was slightly stronger than LSTM in the final comparisons.

---

### 5. Dead ends and weaker settings

Several explored directions did not perform as well:

#### 5.1 Dense baselines
The lecture-aligned Dense baselines with one hidden layer of 32, 64, and 128 nodes were useful reference points, but they were clearly weaker than the recurrent models. Dense_128 was the strongest Dense baseline, but still substantially worse than the GRU champion.

#### 5.2 Conv1D and TCN
Conv1D and TCN captured some temporal structure and performed better than the weaker Dense variants, but they did not match the recurrent models. This suggests that local temporal convolutions alone were less effective than recurrent sequence modeling for this task.

#### 5.3 High dropout
Higher dropout values, especially around 0.2, were generally less effective than very small dropout values such as 0.01. In the final tuning round, the best GRU setup used only **0.01 dropout**, which indicates that mild regularization was better than strong regularization for this dataset.

#### 5.4 Too-large model sweeps too early
A broad search over many models and many dropout settings was computationally expensive and not the most efficient first step once the number of valid sequences increased. A staged process worked better:
1. broad model-family screening
2. reduced family screening
3. focused GRU tuning

---

### 6. Accuracy assessment results

The final reported evaluation is based on **cross-validated mean test MAE**, not on a quick post-training sample check. This is important because the lecture explicitly asks for test-data evaluation and k-fold cross validation.

#### Final champion
**BiGRU_256_2layers_drop001_lr5e4**

#### Cross-validated results
- **Mean test MAE:** 0.03286799360066652
- **Mean test MAE in cm:** **3.2867993600666523 cm**
- **Standard deviation of test MAE in cm:** 0.369940701335222 cm
- **Mean average 2D distance in cm:** **5.443153008818626 cm**
- **Standard deviation of average 2D distance in cm:** 0.6161730291648869 cm

#### Interpretation
The model clearly learned a useful mapping from PoseNet 2D to Kinect 2D, but the cross-validated error remained above the idealized example shown in the lecture. The most likely reasons are:
- remaining PoseNet noise in the generated 2D joint coordinates
- harder distal joints such as hands, elbows, and feet
- residual mismatch between PoseNet and Kinect coordinate systems
- the fact that cross-validated mean test MAE is a stricter and more honest metric than a quick post-training sample check

Thus, the reported result should be interpreted as the best **generalization** performance observed in the notebook, not as a memorization or training-set score.

---

### 7. Software-development outcome

The lecture also requires that the solution be easy to retrain and use.This was addressed by:
- saving the final champion model weights
- saving the input and target scalers
- saving model metadata
- reloading the saved model and running a prediction check

This means the pipeline now supports:
- regenerating PoseNet CSV files
- retraining the champion model
- reloading the saved model for inference

The 3D skeleton visualization/animation part was not the main focus of this notebook and should be treated as a separate software-development extension if it is required in the final submission.

---

### 8. One-step variant `(xpn, ypn) → (xk, yk, zk)`

The lecture suggests also trying a one-step variant that predicts `(xk, yk, zk)` directly from `(xpn, ypn)` with one network.

That variant was **not the main focus of the current notebook**, which concentrated on the required week 16 task of learning Kinect 2D from PoseNet 2D. A natural continuation would be to reuse the current pipeline and:
- keep the same 26 PoseNet inputs
- extend the targets from 26 features to 39 Kinect features
- compare the one-step 3D mapping against the current two-step approach

This is proposed as future work.

---

### 9. Conclusion

The Lab 10 task was implemented as a supervised regression problem that maps PoseNet 2D joint positions to Kinect 2D joint positions. The notebook followed the Deep Learning steps from the lecture: paired data preparation, model definition, compilation, k-fold cross validation, early-stopped training, test-set evaluation in cm, and model reuse.

The experiments showed that:
- sequence-aware models outperform Dense baselines
- GRU is the strongest model family for this task
- very small dropout works better than larger dropout
- the best observed setup was **BiGRU_256_2layers_drop001_lr5e4**

The final cross-validated result of the champion model was:

**Mean test MAE = 3.28 cm**

This is the recommended setup for the current 2D PoseNet to Kinect 2D mapping task.

## SW

### Skeleton Tab Creation
This week in sw we created a new 3D Skeleton Viewer tab in the frontend. The tab lets users upload 3D keypoint CSV files (the test data from last sprint "kinect") and view the motion frame by frame with play, pause, stop, FPS, and frame slider.

The animation is implemented in skeleton-3d.js. The CSV is parsed into frames and normalized so different recordings fit the same viewer space. Each render draws the current frame's joints and bone connections on a canvas, while a interactive projection shows the 3D view on the right side of the tab.


